In [2]:
import pandas as pd
import sqlite3

# Koneksi ke database
conn = sqlite3.connect('../data/cleaned/retail.db')

# Fungsi helper untuk query lebih mudah
def query(sql):
    return pd.read_sql(sql, conn)

print("Koneksi ke retail.db berhasil")

Koneksi ke retail.db berhasil


In [3]:
# Query total revenue per negara

result = query("""
    SELECT 
        Country,
        ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers
    FROM retail
    GROUP BY Country
    ORDER BY Total_Revenue DESC
    LIMIT 10
""")

result

,Country,Total_Revenue,Total_Orders,Total_Customers
0,United Kingdom,14389234.92,33541,5350
1,EIRE,616570.54,567,5
2,Netherlands,554038.09,228,22
3,Germany,425019.71,789,107
4,France,348768.96,614,95
5,Australia,169283.46,95,15
6,Spain,108332.49,154,41
7,Switzerland,100061.94,90,22
8,Sweden,91515.82,104,19
9,Denmark,68580.69,43,12


In [4]:
# Query monthly sales trend

result = query("""
    SELECT
        Year,
        Month,
        ROUND(SUM(TotalPrice), 2) AS Monthly_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers
    FROM retail
    GROUP BY Year, Month
    ORDER BY Year, Month
""")

result

,Year,Month,Monthly_Revenue,Total_Orders,Total_Customers
0,2009,12,683504.01,1512,955
1,2010,1,555802.67,1011,720
2,2010,2,504558.96,1104,772
3,2010,3,696978.47,1524,1057
4,2010,4,591982.00,1329,942
5,2010,5,597833.38,1377,966
6,2010,6,636371.13,1497,1041
7,2010,7,589736.17,1381,928
8,2010,8,602224.60,1293,911
9,2010,9,829013.95,1689,1145


In [ ]:
# Query untuk top 10 produk terlaris (total_quantity_Sold)

result = query("""
    SELECT
        StockCode,
        Description,
        SUM(Quantity) AS Total_Quantity_Sold,
        ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders
    FROM retail
    GROUP BY StockCode, Description
    ORDER BY Total_Quantity_Sold DESC
    LIMIT 10
""")

result

,StockCode,Description,Total_Quantity_Sold,Total_Revenue,Total_Orders
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,105185,24098.03,920
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,91757,247048.01,4888
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
3,84879,ASSORTED COLOUR BIRD ORNAMENT,78234,124351.86,2652
4,23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,195
5,85099B,JUMBO BAG RED RETROSPOT,74224,134307.44,2612
6,17003,BROCADE RING PURSE,70082,14640.96,387
7,21977,PACK OF 60 PINK PAISLEY CAKE CASES,54592,26407.35,1578
8,84991,60 TEATIME FAIRY CAKE CASES,52828,25785.92,1765
9,21212,PACK OF 72 RETRO SPOT CAKE CASES,45129,21838.23,1160


In [6]:
# Query untuk frekuensi transaksi per customer

result = query("""
    SELECT
        "Customer ID",
        COUNT(DISTINCT Invoice) AS Total_Orders,
        ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
        ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value,
        MIN(InvoiceDate) AS First_Purchase,
        MAX(InvoiceDate) AS Last_Purchase
    FROM retail
    GROUP BY "Customer ID"
    ORDER BY Total_Revenue DESC
    LIMIT 10
""")

result

,Customer ID,Total_Orders,Total_Revenue,Avg_Order_Value,First_Purchase,Last_Purchase
0,18102,145,580987.04,558.64,2009-12-01 09:24:00,2011-12-09 11:50:00
1,14646,151,528602.52,137.34,2009-12-02 16:52:00,2011-12-08 12:12:00
2,14156,156,313437.62,77.62,2009-12-01 12:30:00,2011-11-30 10:54:00
3,14911,398,291420.81,26.31,2009-12-01 11:41:00,2011-12-08 15:54:00
4,17450,51,244784.25,582.82,2010-09-27 16:59:00,2011-12-01 13:29:00
5,13694,143,195640.69,128.80,2009-12-04 15:26:00,2011-12-06 09:32:00
6,17511,60,172132.87,92.15,2009-12-02 10:52:00,2011-12-07 10:12:00
7,16446,2,168472.50,56157.50,2011-05-18 09:52:00,2011-12-09 09:15:00
8,16684,55,147142.77,204.93,2009-12-07 12:56:00,2011-12-05 14:06:00
9,12415,28,144458.37,156.00,2010-06-30 08:30:00,2011-11-15 14:22:00


In [11]:
# Query untuk cancellation rate per negara

result = query("""
    SELECT
        Country,
        COUNT(DISTINCT CASE WHEN is_cancelled = 0 
            THEN Invoice END) AS Total_Orders,
        COUNT(DISTINCT CASE WHEN is_cancelled = 1 
            THEN Invoice END) AS Total_Cancellations,
        ROUND(COUNT(DISTINCT CASE WHEN is_cancelled = 1 THEN Invoice END) * 100.0 /
              NULLIF(COUNT(DISTINCT CASE WHEN is_cancelled = 0 
                THEN Invoice END), 0), 2) AS Cancellation_Rate
    FROM (
        SELECT Invoice, Country, 0 AS is_cancelled FROM retail
        UNION ALL
        SELECT Invoice, Country, 1 AS is_cancelled FROM cancelled
    )
    GROUP BY Country
    ORDER BY Cancellation_Rate DESC
    LIMIT 10
""")

result

,Country,Total_Orders,Total_Cancellations,Cancellation_Rate
0,Czech Republic,2,3,150.00
1,Saudi Arabia,1,1,100.00
2,Lebanon,1,1,100.00
3,Malta,9,7,77.78
4,Japan,33,23,69.70
5,United Arab Emirates,11,6,54.55
6,RSA,2,1,50.00
7,Nigeria,2,1,50.00
8,Korea,2,1,50.00
9,Bahrain,4,2,50.00


In [8]:
# Query untuk revenue per bulan & tahun

result = query("""
    SELECT
        Year,
        Month,
        ROUND(SUM(TotalPrice), 2) AS Monthly_Revenue,
        ROUND(SUM(TotalPrice) - LAG(SUM(TotalPrice)) 
              OVER (ORDER BY Year, Month), 2) AS Revenue_Change,
        ROUND((SUM(TotalPrice) - LAG(SUM(TotalPrice)) 
              OVER (ORDER BY Year, Month)) * 100.0 / 
              LAG(SUM(TotalPrice)) OVER (ORDER BY Year, Month), 2) AS MoM_Growth
    FROM retail
    GROUP BY Year, Month
    ORDER BY Year, Month
""")

result

,Year,Month,Monthly_Revenue,Revenue_Change,MoM_Growth
0,2009,12,683504.01,NaN,NaN
1,2010,1,555802.67,-127701.34,-18.68
2,2010,2,504558.96,-51243.72,-9.22
3,2010,3,696978.47,192419.52,38.14
4,2010,4,591982.00,-104996.47,-15.06
5,2010,5,597833.38,5851.38,0.99
6,2010,6,636371.13,38537.75,6.45
7,2010,7,589736.17,-46634.96,-7.33
8,2010,8,602224.60,12488.43,2.12
9,2010,9,829013.95,226789.35,37.66


In [9]:
# Query untuk AOV per bulan

result = query("""
    SELECT
        Year,
        Month,
        ROUND(SUM(TotalPrice) / COUNT(DISTINCT Invoice), 2) AS AOV,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers
    FROM retail
    GROUP BY Year, Month
    ORDER BY Year, Month
""")

result

,Year,Month,AOV,Total_Orders,Total_Customers
0,2009,12,452.05,1512,955
1,2010,1,549.76,1011,720
2,2010,2,457.03,1104,772
3,2010,3,457.33,1524,1057
4,2010,4,445.43,1329,942
5,2010,5,434.16,1377,966
6,2010,6,425.10,1497,1041
7,2010,7,427.04,1381,928
8,2010,8,465.76,1293,911
9,2010,9,490.83,1689,1145


In [10]:
# Query untuk summary keseluruhan

result = query("""
    SELECT
        ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers,
        COUNT(DISTINCT StockCode) AS Total_Products,
        COUNT(DISTINCT Country) AS Total_Countries,
        ROUND(SUM(TotalPrice) / COUNT(DISTINCT Invoice), 2) AS AOV
    FROM retail
""")

result

,Total_Revenue,Total_Orders,Total_Customers,Total_Products,Total_Countries,AOV
0,17374804.27,36969,5878,4631,41,469.98


In [ ]:
# Simpan hasil summary untuk retail_analytics.ipynb
kpi = query("""
    SELECT
        ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers,
        COUNT(DISTINCT StockCode) AS Total_Products,
        COUNT(DISTINCT Country) AS Total_Countries,
        ROUND(SUM(TotalPrice) / COUNT(DISTINCT Invoice), 2) AS AOV
    FROM retail
""")
kpi.to_csv('../data/cleaned/kpi_summary.csv', index=False)

# Simpan monthly trend untuk dipakai di EDA & dashboard
monthly = query("""
    SELECT
        Year, Month,
        ROUND(SUM(TotalPrice), 2) AS Monthly_Revenue,
        COUNT(DISTINCT Invoice) AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers
    FROM retail
    GROUP BY Year, Month
    ORDER BY Year, Month
""")
monthly.to_csv('../data/cleaned/monthly_trend.csv', index=False)

# Tutup koneksi
conn.close()

print("kpi_summary.csv berhasil disimpan.")
print("monthly_trend.csv berhasil disimpan.")
print("Koneksi database ditutup.")

✅ kpi_summary.csv berhasil disimpan!
✅ monthly_trend.csv berhasil disimpan!
✅ Koneksi database ditutup!
